## Messge History
We can use Message History class to wrap our model and make it stateful. This will keep track of input and output of the model, and store them in some datastore. Further interactions will then load those messages and pass them into the chain as part of the input.

In [8]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
groq_api_key

from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
model = ChatGroq(model="gemma2-9b-it", groq_api_key=groq_api_key)

In [9]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


store={}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]
with_message_history=RunnableWithMessageHistory(model, get_session_history)

In [10]:
config = {"configurable": {"session_id":"session_id_1"}}

In [11]:
from langchain_core.messages import HumanMessage, SystemMessage
with_message_history.invoke(
    [HumanMessage(content="Hello, My name is Nadeem and I am AI Engineer")],
    config=config

)

AIMessage(content="Hello Nadeem, it's nice to meet you! 👋\n\nAs an AI myself, I'm always interested in connecting with other people in the field. What kind of AI engineering work do you do?  \n\nPerhaps you could tell me about:\n\n* **Your current projects:** What are you working on that you're excited about?\n* **Your area of expertise:** Do you specialize in a particular area of AI, like machine learning, natural language processing, or computer vision?\n* **Your favorite tools or technologies:** What are some of your go-to resources for AI development?\n\n\nI'm eager to learn more about your work! 😊\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 139, 'prompt_tokens': 21, 'total_tokens': 160, 'completion_time': 0.252727273, 'prompt_time': 0.00132437, 'queue_time': 0.186919201, 'total_time': 0.254051643}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='r

In [12]:
with_message_history.invoke(
    [HumanMessage(content="What is my name")],
    config=config

)

AIMessage(content='Your name is Nadeem, as you told me at the beginning of our conversation.  😄 \n\n\nIs there anything else I can help you with?\n', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 172, 'total_tokens': 206, 'completion_time': 0.061818182, 'prompt_time': 0.004458959, 'queue_time': 0.18804791799999998, 'total_time': 0.066277141}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--5ce72d6c-4c7c-4416-9c46-d63d54bf2b1b-0', usage_metadata={'input_tokens': 172, 'output_tokens': 34, 'total_tokens': 206})

In [13]:
# Change the config --> session_id
config = {"configurable": {"session_id":"session_id_2"}}
with_message_history.invoke(
    [HumanMessage(content="What is my name")],
    config=config

)

AIMessage(content="As an AI, I have no memory of past conversations and do not know your name. If you'd like to tell me your name, I'd be happy to use it!\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 13, 'total_tokens': 54, 'completion_time': 0.074545455, 'prompt_time': 0.00126907, 'queue_time': 0.18925283399999998, 'total_time': 0.075814525}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--a35449ea-53ab-4227-a0b6-e8198b2e7627-0', usage_metadata={'input_tokens': 13, 'output_tokens': 41, 'total_tokens': 54})

Prompmpt Template

Prompt Template help to turn raw user information into a formate that LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM.

In [14]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompts=ChatPromptTemplate.from_messages(
    [
        ("system", "You are helpful assistant. Answer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="message")
    ]
)
chain=prompts|model

In [28]:
from langchain_core.messages import HumanMessage
chain.invoke({"message": [HumanMessage(content="Hallo, my name is Nadeem")]})

AIMessage(content="Hello Nadeem! It's nice to meet you.  \n\nI'm ready to answer your questions. Just ask away! I'll do my best to be helpful and provide you with the information you need. 😊  \n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 31, 'total_tokens': 81, 'completion_time': 0.090909091, 'prompt_time': 0.001483608, 'queue_time': 0.187646855, 'total_time': 0.092392699}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--4b110a28-e603-43e5-b874-1c42a61a831c-0', usage_metadata={'input_tokens': 31, 'output_tokens': 50, 'total_tokens': 81})

In [29]:
with_message_history=RunnableWithMessageHistory(chain, get_session_history)

In [31]:
config = {"configurable": {"session_id":"session_id_3"}}
response = with_message_history.invoke(
    [HumanMessage(content="My Name is Nadeem")],
    config=config
)
response.content

"Hi Nadeem, it's nice to meet you! 😊\n\nIs there anything else I can help you with?  \n"

In [32]:
response = with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config
)
response.content

'You told me your name is Nadeem!  \n\nHow can I help you further? 😄  \n'

Managing the Conversation History

One important concept to understand when building chatbots is to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. 

Trim Message: It help to reduce how many message we are sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameter like if we want to always keep the system message and wheather to allow partial messages.


In [38]:
from langchain_core.messages import SystemMessage, trim_messages
trimmer=trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on='human'
)

messages= [
    SystemMessage(content="You are a good assistant"),
    HumanMessage(content="Hi I am BOB"),
    SystemMessage(content="Hello"),
    HumanMessage(content="I like Vanilla Ice cream"),
    SystemMessage(content="Nice"),
    HumanMessage(content="what is 2+2"),
    SystemMessage(content="4")
]

In [39]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough
chain=(
    RunnablePassthrough.assign(message=itemgetter("messages")|trimmer) | prompts | model
)
response = chain.invoke(
    {
         'messages': messages + [HumanMessage(content="What ice cream do I like")],
         "language": "English"

    }
)
response.content

ImportError: Could not import transformers python package. This is needed in order to calculate get_token_ids. Please install it with `pip install transformers`.